# Refactored train/test split pipeline
This notebook orchestrates the original `train_test_split.ipynb` workflow using code extracted into the `src` package.


In [ ]:
import os
import sys
import numpy as np

# Allow imports from the project root
sys.path.insert(0, os.path.abspath('..'))

from src.data_prep import prepare_data
from src import regression
from src.pipeline import FEATURE_SETS, plot_feature_histograms, reset_output_dir, run_baseline_regressions
from src.model_comparison import run_model_comparison
from src.classification import run_classification_analysis

import matplotlib
matplotlib.use("Agg")

In [ ]:
if os.path.exists('outputs/paper_plots') is False:
    os.makedirs('outputs/paper_plots')

In [ ]:
# Configuration flags and parameters
RUN_BASELINE_MODELS = True
RUN_BASELINE_MIXED = True
RUN_BASELINE_DOMAIN_SHIFT = True
BASELINE_MIXED_LABEL = 'Mixed split'
BASELINE_DOMAIN_LABEL = 'E-INSPIRE→INSPIRE'
BASELINE_PLOTTING = False
RANDOM_STATES = [1, 2, 3, 4, 5, 6, 7, 8, 9, 1000000]

RUN_HISTOGRAMS = True
RUN_CORNER = False
RUN_CORNER_BY_DATASET = True
RUN_RESIDUALS = False
RUN_CLASSIFICATION = True
RUN_MODEL_COMPARISON = True

MODEL_COMPARISON_SEEDS = [1,2,3,4,5]
MODEL_COMPARISON_MODES = [
    # ('mixed', 'Mixed split'),
    ('domain_shift', 'E-INSPIRE→INSPIRE'),
]
CLASSIFICATION_THRESHOLDS = np.arange(0.25, 0.66, 0.05)
CLASSIFICATION_FEATURES = ['tau', 'lin_age_err', 'met', 'met_err', 'logM', 'rad_kpc']
CLASSIFICATION_DATASET_MODES = [
    ('mixed', 'Mixed split'),
    ('domain_shift', 'E-INSPIRE→INSPIRE'),
]

CORNER_FEATURES = ['met', 'tau', 'met_err', 'lin_age_err', 'logM', 'rad_kpc', 'MgFe', 'vdisp', 'DoR']
CORNER_TARGET_DISPLAY_NAME = r'$\mathrm{DoR}$'

RESIDUAL_FEATURES = ['tau', 'met']
RESIDUAL_MODEL_PARAMS = {
    'max_depth': 8,
    'max_features': 0.8,
    'max_samples': 0.7,
    'min_samples_leaf': 3,
    'min_samples_split': 5,
    'n_estimators': 50,
    'random_state': 42,
}


In [ ]:
columns = ['vdisp','tau','MgFe', 'met_err', 'lin_age_err','met','rad_kpc','logM','DoR']
mixed_train_df, mixed_test_df = prepare_data(columns, restricted=False, pc=False, mix_datasets=True)
domain_train_df, domain_test_df = prepare_data(columns, restricted=False, pc=False, mix_datasets=False)

# Default references for downstream plots (mixed split)
train_df, test_df = mixed_train_df, mixed_test_df
regression.train_df = train_df
regression.test_df = test_df

# Unmixed data for E-INSPIRE vs INSPIRE visualisations
hist_train_df, hist_test_df = domain_train_df, domain_test_df


In [ ]:
# Prepare output directory used throughout the pipeline
reset_output_dir('outputs/tests')


In [ ]:
if RUN_BASELINE_MODELS:
    if RUN_BASELINE_MIXED:
        run_baseline_regressions(mixed_train_df, mixed_test_df, RANDOM_STATES, plotting=BASELINE_PLOTTING, tag=BASELINE_MIXED_LABEL)
    if RUN_BASELINE_DOMAIN_SHIFT:
        run_baseline_regressions(domain_train_df, domain_test_df, RANDOM_STATES, plotting=BASELINE_PLOTTING, tag=BASELINE_DOMAIN_LABEL)
    regression.print_results_summary()
    ensemble_mixed = regression.calculate_ensemble_metrics(
        dataset_frames={
            BASELINE_MIXED_LABEL: (mixed_train_df, mixed_test_df),
            BASELINE_DOMAIN_LABEL: (domain_train_df, domain_test_df),
        },
        filter_tags=[BASELINE_MIXED_LABEL],
        output_path='outputs/paper_plots/ensemble_results_mixed.csv',
    )
    ensemble_domain = regression.calculate_ensemble_metrics(
        dataset_frames={
            BASELINE_MIXED_LABEL: (mixed_train_df, mixed_test_df),
            BASELINE_DOMAIN_LABEL: (domain_train_df, domain_test_df),
        },
        filter_tags=[BASELINE_DOMAIN_LABEL],
        output_path='outputs/paper_plots/ensemble_results_domain_shift.csv',
    )


In [ ]:
if RUN_HISTOGRAMS:
    # Unmixed E-INSPIRE vs INSPIRE
    plot_feature_histograms(hist_train_df, hist_test_df, tag='unmixed')
    # Mixed train/test split
    plot_feature_histograms(mixed_train_df, mixed_test_df, tag='mixed')


In [ ]:
if RUN_CORNER:
    regression.create_corner_plots(
        hist_test_df,
        CORNER_FEATURES,
        target='DoR',
        target_display_name=CORNER_TARGET_DISPLAY_NAME,
    )
    regression.create_corner_plots(
        hist_train_df,
        CORNER_FEATURES,
        target='DoR',
        target_display_name=CORNER_TARGET_DISPLAY_NAME,
    )


In [ ]:
if RUN_CORNER_BY_DATASET:
    feature_display_names = {
        'met': r'$\mathrm{[M/H]}$',
        'tau': r'$\mathrm{\tau_{\rm rel}}$',
        'met_err': r'$\Delta{\mathrm{[M/H]}}$',
        'lin_age_err': r'$\Delta{\mathrm{Age}}$',
        'logM': r'$\log(M/M_{\odot})$',
        'rad_kpc': r'$R \, \mathrm{(kpc)}$',
        'MgFe': r'$\mathrm{[Mg/Fe]}$',
        'vdisp': r'$\sigma \star \mathrm{(km/s)}$',
        'DoR': r'$\mathrm{DoR}$',
    }
    regression.plot_features_by_dataset(
        hist_train_df,
        hist_test_df,
        CORNER_FEATURES,
        feature_display_names=feature_display_names,
    )


In [ ]:
if RUN_RESIDUALS:
    regression.create_residual_analysis_report(
        train_df,
        test_df,
        RESIDUAL_FEATURES,
        'DoR',
        RESIDUAL_MODEL_PARAMS,
    )


In [ ]:
if RUN_CLASSIFICATION:
    for dataset_mode, dataset_label in CLASSIFICATION_DATASET_MODES:
        cv_results = run_classification_analysis(CLASSIFICATION_THRESHOLDS, CLASSIFICATION_FEATURES, dataset_mode=dataset_mode, dataset_label=dataset_label)


In [ ]:
MODEL_COMPARISON_SEEDS = [3,4,5,6,7]
if RUN_MODEL_COMPARISON:
    for dataset_mode, dataset_label in MODEL_COMPARISON_MODES:
        results_df, agg_results = run_model_comparison(MODEL_COMPARISON_SEEDS, True, dataset_mode=dataset_mode, dataset_label=dataset_label)
    print('Model comparison completed and results saved to outputs/tests/')
